# 구조물 안정성 예측 - Colab A100 학습

## 사전 준비
1. Google Drive에 프로젝트 zip 파일 업로드
2. 런타임 → 런타임 유형 변경 → **A100 GPU** 선택
3. 셀을 순서대로 실행

## 구조
- **Cell 1**: 환경 설정 (Drive 마운트, zip 해제, 패키지 설치)
- **Cell 2**: GPU 확인 & batch size 자동 설정
- **Cell 3**: Pretrain (efficientnet / convnext / swinv2)
- **Cell 4**: Finetune (5-Fold CV)
- **Cell 5**: Inference & submission 생성
- **Cell 6**: 결과물 Drive로 백업
- **Cell 7**: 로컬 체크포인트 복원

In [ ]:
#@title [Cell 1] 환경 설정 (Drive 마운트 + zip 해제 + 패키지 설치)
#@markdown ---
#@markdown **Google Drive 내 zip 파일 경로** (절대 경로)
ZIP_PATH_IN_DRIVE = "/content/drive/MyDrive/개인_공부방/DACON/data/data.zip" #@param {type:"string"}
#@markdown **Drive 프로젝트 폴더** (코드 + 체크포인트 저장용)
DRIVE_PROJECT = "/content/drive/MyDrive/개인_공부방/DACON" #@param {type:"string"}
#@markdown ---

import os, shutil, subprocess, sys, zipfile

# === Colab 로컬 SSD 작업 디렉토리 (빠른 I/O) ===
WORK_DIR = "/content/dacon-structural-stability"

# 1) Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

assert os.path.exists(ZIP_PATH_IN_DRIVE), f"ZIP 파일을 찾을 수 없습니다: {ZIP_PATH_IN_DRIVE}"
zip_size_gb = os.path.getsize(ZIP_PATH_IN_DRIVE) / 1e9
print(f"[OK] ZIP 발견: {ZIP_PATH_IN_DRIVE} ({zip_size_gb:.2f} GB)")

# 2) 코드 파일을 Drive에서 Colab 로컬로 복사 (빠른 SSD 사용)
if os.path.exists(WORK_DIR) and os.path.exists(os.path.join(WORK_DIR, 'train.py')):
    print(f"[SKIP] 기존 작업 디렉토리 재사용: {WORK_DIR}")
else:
    os.makedirs(WORK_DIR, exist_ok=True)
    # 코드 파일 복사 (Drive -> 로컬 SSD)
    for f in ['train.py', 'datasets.py', 'models.py', 'inference.py',
              'requirements.txt', 'download_shapestacks.py']:
        src = os.path.join(DRIVE_PROJECT, f)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(WORK_DIR, f))
    print(f"[OK] 코드 복사 완료: {WORK_DIR}")

# 3) 데이터 zip 해제 (Python zipfile 사용 -- ZIP64 대용량 지원)
TARGET_DATA_DIR = os.path.join(WORK_DIR, 'data')
if os.path.exists(os.path.join(TARGET_DATA_DIR, 'open', 'train.csv')):
    print(f"[SKIP] 데이터 이미 존재, 재사용: {TARGET_DATA_DIR}")
else:
    print(f"[...] ZIP 해제 중... (Python zipfile, ZIP64 지원)")
    os.makedirs(TARGET_DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH_IN_DRIVE, 'r') as zf:
        total = len(zf.namelist())
        for i, member in enumerate(zf.namelist()):
            zf.extract(member, TARGET_DATA_DIR)
            if (i + 1) % 10000 == 0 or (i + 1) == total:
                print(f"  [{i+1}/{total}] 해제 중...")
    print(f"[OK] 해제 완료: {TARGET_DATA_DIR}")
    # 해제 결과 확인
    for root, dirs, files in os.walk(TARGET_DATA_DIR):
        level = root.replace(TARGET_DATA_DIR, '').count(os.sep)
        if level < 2:
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print(f"\n[DIR] 작업 디렉토리: {os.getcwd()}")
print(f"[FILES] {[f for f in os.listdir('.') if not f.startswith('.')]}")

# 4) 패키지 설치
print("\n[...] 패키지 설치 중...")
!pip install -q timm albumentations opencv-python-headless scikit-learn tqdm av

# 5) 체크포인트는 Drive에 저장 (런타임 종료되어도 유지)
# 로컬 checkpoints -> Drive symlink
DRIVE_CKPT = os.path.join(DRIVE_PROJECT, 'checkpoints')
LOCAL_CKPT = os.path.join(WORK_DIR, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)
if os.path.islink(LOCAL_CKPT):
    os.unlink(LOCAL_CKPT)
elif os.path.exists(LOCAL_CKPT):
    # 기존 체크포인트 Drive로 이동
    for f in os.listdir(LOCAL_CKPT):
        shutil.move(os.path.join(LOCAL_CKPT, f), DRIVE_CKPT)
    os.rmdir(LOCAL_CKPT)
os.symlink(DRIVE_CKPT, LOCAL_CKPT)
print(f"[OK] checkpoints -> Drive symlink ({DRIVE_CKPT})")

os.makedirs(os.path.join(WORK_DIR, 'submissions'), exist_ok=True)

# 6) data 디렉토리 확인
if os.path.exists('data/open/train.csv'):
    import pandas as pd
    n_train = len(pd.read_csv('data/open/train.csv'))
    n_dev = len(pd.read_csv('data/open/dev.csv')) if os.path.exists('data/open/dev.csv') else 0
    print(f"[OK] data/open (train: {n_train}, dev: {n_dev})")
if os.path.exists('data/archive/blocks-labels.csv'):
    print("[OK] data/archive")
if os.path.exists('data/shapestacks'):
    print("[OK] data/shapestacks")
else:
    print("[WARN] data/shapestacks 없음 - ShapeStacks 없이 학습 진행")

# 디스크 사용량 확인
!df -h /content | tail -1 | awk '{print "[DISK] 사용:", $3, "/ 전체:", $2, "/ 남은:", $4}'

print("\n=== 환경 설정 완료! ===")

In [ ]:
#@title [Cell 2] GPU 확인 & A100 최적 설정
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")

    # A100 최적 설정
    if 'A100' in gpu_name:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("[OK] A100 TF32 활성화 (속도 UP, 정밀도 유지)")
        BATCH_MULTIPLIER = 4  # A100 80GB -> batch 4배
    elif 'V100' in gpu_name:
        BATCH_MULTIPLIER = 2
    elif 'T4' in gpu_name:
        BATCH_MULTIPLIER = 1
    elif 'L4' in gpu_name:
        BATCH_MULTIPLIER = 2
    else:
        BATCH_MULTIPLIER = 2

    print(f"Batch multiplier: x{BATCH_MULTIPLIER}")
else:
    print("[ERROR] GPU 없음! 런타임 유형을 확인하세요.")
    BATCH_MULTIPLIER = 1

In [ ]:
#@title [Cell 3] Pretrain
#@markdown ---
MODEL = "efficientnet" #@param ["efficientnet", "convnext", "swinv2"]
PRETRAIN_EPOCHS = 30 #@param {type:"integer"}
RESUME = True #@param {type:"boolean"}
MAX_ARCHIVE = 10000 #@param {type:"integer"}
MAX_SHAPESTACKS = 0 #@param {type:"integer"}
#@markdown `MAX_SHAPESTACKS=0` -> 전부 사용
#@markdown ---

import os, sys, importlib
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import train as train_module
importlib.reload(train_module)

# A100용 batch size 패치 (원본 dict 변경 없이 복사본 반환)
_orig_get_config = train_module.get_model_config
def _patched_get_config(model_type):
    cfg = _orig_get_config(model_type).copy()
    cfg['batch_size'] = cfg['batch_size'] * BATCH_MULTIPLIER
    return cfg
train_module.get_model_config = _patched_get_config

orig_config = _orig_get_config(MODEL)
print(f"Batch size: {orig_config['batch_size']} -> {orig_config['batch_size'] * BATCH_MULTIPLIER} (x{BATCH_MULTIPLIER})")

# argparse 우회
class Args:
    model = MODEL
    stage = 'pretrain'
    pretrain_epochs = PRETRAIN_EPOCHS
    finetune_epochs = 50
    n_folds = 5
    fold = None
    patience = 10
    seed = 42
    max_archive_samples = MAX_ARCHIVE
    max_shapestacks_samples = MAX_SHAPESTACKS
    resume = RESUME

args = Args()
print(f"\n{'='*60}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Model: {MODEL} | Epochs: {PRETRAIN_EPOCHS} | Resume: {RESUME}")
print(f"{'='*60}\n")

train_module.pretrain(args)

# 복원
train_module.get_model_config = _orig_get_config
print("\n=== Pretrain 완료! ===")

In [ ]:
#@title [Cell 4] Finetune (5-Fold CV)
#@markdown ---
MODEL = "efficientnet" #@param ["efficientnet", "convnext", "swinv2"]
FINETUNE_EPOCHS = 50 #@param {type:"integer"}
PATIENCE = 10 #@param {type:"integer"}
SPECIFIC_FOLD = None #@param {type:"raw"}
RESUME = True #@param {type:"boolean"}
#@markdown `SPECIFIC_FOLD`: None=전체, 0~4=특정 fold
#@markdown ---

import os, sys, importlib
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import train as train_module
importlib.reload(train_module)

# A100용 batch size 패치 (복사본 반환)
_orig_get_config = train_module.get_model_config
def _patched_get_config(model_type):
    cfg = _orig_get_config(model_type).copy()
    cfg['batch_size'] = cfg['batch_size'] * BATCH_MULTIPLIER
    return cfg
train_module.get_model_config = _patched_get_config

class Args:
    model = MODEL
    stage = 'finetune'
    pretrain_epochs = 30
    finetune_epochs = FINETUNE_EPOCHS
    n_folds = 5
    fold = SPECIFIC_FOLD
    patience = PATIENCE
    seed = 42
    max_archive_samples = 10000
    max_shapestacks_samples = 0
    resume = RESUME

args = Args()
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Model: {MODEL} | Epochs: {FINETUNE_EPOCHS} | Patience: {PATIENCE} | Resume: {RESUME}")

train_module.finetune(args)

train_module.get_model_config = _orig_get_config
print("\n=== Finetune 완료! ===")

In [ ]:
#@title [Cell 5] Inference & Submission
#@markdown ---
USE_TTA = True #@param {type:"boolean"}
SINGLE_MODEL = "" #@param {type:"string"}
#@markdown `SINGLE_MODEL`: 비우면 전체 앙상블, 입력하면 단일 모델 (efficientnet/convnext/swinv2)
#@markdown ---

import os
os.chdir(WORK_DIR)

cmd = "python inference.py"
if USE_TTA:
    cmd += " --tta"
if SINGLE_MODEL:
    cmd += f" --model {SINGLE_MODEL}"

print(f"실행: {cmd}\n")
!{cmd}

# 결과 확인
import pandas as pd
sub_dir = 'submissions'
if os.path.exists(sub_dir):
    subs = sorted([f for f in os.listdir(sub_dir) if f.endswith('.csv')])
    if subs:
        latest = os.path.join(sub_dir, subs[-1])
        df = pd.read_csv(latest)
        print(f"\n[RESULT] 최신 submission: {subs[-1]}")
        print(f"  샘플 수: {len(df)}")
        print(f"  stable 비율: {(df.iloc[:, 1] > 0.5).mean():.2%}")
        print(df.head())

In [ ]:
#@title [Cell 6] submission Drive 백업
#@markdown ---
#@markdown 체크포인트는 symlink로 이미 Drive에 저장됨. submission만 백업.
#@markdown ---

import shutil, os, glob
os.chdir(WORK_DIR)

# submission 백업
sub_src = 'submissions'
sub_dst = os.path.join(DRIVE_PROJECT, 'submissions')
os.makedirs(sub_dst, exist_ok=True)

copied = 0
if os.path.exists(sub_src):
    for f in glob.glob(f'{sub_src}/*.csv'):
        shutil.copy2(f, os.path.join(sub_dst, os.path.basename(f)))
        print(f"  [CSV] {os.path.basename(f)}")
        copied += 1

# 체크포인트 현황
ckpt_files = glob.glob('checkpoints/*.pth')
print(f"\n[CKPT] 체크포인트 ({len(ckpt_files)}개, Drive에 자동 저장):")
for f in ckpt_files:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.0f} MB)")

print(f"\n[OK] {copied}개 submission 백업 완료")

In [ ]:
#@title [Cell 7] 체크포인트 확인 (로컬에서 이어서 할 때)
#@markdown ---
#@markdown Drive의 체크포인트가 symlink로 연결되어 있으므로 별도 복원 불필요.
#@markdown Cell 1을 실행하면 자동으로 symlink이 다시 연결됩니다.
#@markdown 이후 Cell 3/4에서 `RESUME=True`로 이어서 학습하세요.
#@markdown ---

import os, glob
os.chdir(WORK_DIR)

ckpt_dir = 'checkpoints'
if os.path.islink(ckpt_dir):
    target = os.readlink(ckpt_dir)
    files = glob.glob(f'{ckpt_dir}/*.pth')
    print(f"[OK] checkpoints -> {target}")
    for f in files:
        print(f"  * {os.path.basename(f)} ({os.path.getsize(f)/1e6:.0f} MB)")
    if not files:
        print("  (체크포인트 없음 - 처음부터 학습)")
else:
    print("[WARN] symlink 없음. Cell 1을 먼저 실행하세요.")

## 사용 가이드

### 처음 학습
1. Cell 1 > Cell 2 > Cell 3 (pretrain) > Cell 4 (finetune) > Cell 5 (inference) > Cell 6 (백업)

### 로컬에서 이어서 / 런타임 재시작
1. Cell 1 > Cell 2 > Cell 7 (체크포인트 확인) > Cell 3 or 4 (`RESUME=True`)

### 3모델 전부 학습
1. Cell 3에서 `MODEL` 바꿔가며 3번 실행 (efficientnet > convnext > swinv2)
2. Cell 4에서 동일하게 3번 finetune
3. Cell 5에서 `SINGLE_MODEL` 비워두면 3모델 앙상블

### A100 배치 사이즈
| 모델 | 로컬 (8GB) | A100 (80GB) |
|------|-----------|-------------|
| EfficientNetV2-M | 8 | 32 |
| ConvNeXt-Base | 4 | 16 |
| SwinV2-Small | 8 | 32 |

### 폴더 구조
```
/content/dacon-structural-stability/     <-- Colab 로컬 SSD (빠른 I/O)
  +-- train.py, datasets.py, ...         <-- 코드
  +-- data/                              <-- 데이터 (zip에서 해제)
  +-- checkpoints/ -> Drive symlink      <-- 체크포인트 (런타임 종료되어도 유지)
```